In [2]:
import secrets
from dotenv import load_dotenv, find_dotenv
from doc_chat.rag.document_loader import DocumentLoader
from doc_chat.rag.vector_store import VectorStore
import chromadb

_ = load_dotenv(find_dotenv())


In [3]:
file_path = "../data/pdf/The Hundred-Page Machine Learning Book.pdf"
doc_loader = DocumentLoader(file_path)
documents, splits = doc_loader.load_and_split()
print("Number of documents: " + str(len(documents)))
print("Number of splits: " + str(len(splits)))

Number of documents: 152
Number of splits: 385


In [4]:
# document format
documents[0]

Document(metadata={'producer': '3-Heights(TM) PDF Optimization Shell 4.8.25.2 (http://www.pdf-tools.com)', 'creator': 'PyPDF', 'creationdate': '2018-12-18T05:07:46+00:00', 'moddate': '2019-01-22T19:51:34+00:00', 'source': '../data/pdf/The Hundred-Page Machine Learning Book.pdf', 'total_pages': 152, 'page': 0, 'page_label': '1'}, page_content='The\nHundred-\nPage\nMachine\nLearning\nBook\nAndriy Burkov')

In [5]:
# print first 300 characters of the third page
documents[2].page_content[:300]

'Preface\nLet’s start by telling the truth: machines don’t learn. What a typical “learning machine”\ndoes, is ﬁnding a mathematical formula, which, when applied to a collection of inputs (called\n“training data”), produces the desired outputs. This mathematical formula also generates the\ncorrect outputs'

In [6]:
# split format
splits[0:2]

[Document(metadata={'producer': '3-Heights(TM) PDF Optimization Shell 4.8.25.2 (http://www.pdf-tools.com)', 'creator': 'PyPDF', 'creationdate': '2018-12-18T05:07:46+00:00', 'moddate': '2019-01-22T19:51:34+00:00', 'source': '../data/pdf/The Hundred-Page Machine Learning Book.pdf', 'total_pages': 152, 'page': 0, 'page_label': '1'}, page_content='The\nHundred-\nPage\nMachine\nLearning\nBook\nAndriy Burkov'),
 Document(metadata={'producer': '3-Heights(TM) PDF Optimization Shell 4.8.25.2 (http://www.pdf-tools.com)', 'creator': 'PyPDF', 'creationdate': '2018-12-18T05:07:46+00:00', 'moddate': '2019-01-22T19:51:34+00:00', 'source': '../data/pdf/The Hundred-Page Machine Learning Book.pdf', 'total_pages': 152, 'page': 1, 'page_label': '2'}, page_content='“All models are wrong, but some are useful.”\n— George Box\nThe book is distributed on the “read ﬁrst, buy later” principle.\nAndriy Burkov The Hundred-Page Machine Learning Book - Draft')]

In [36]:
splits[2].page_content

'Preface\nLet’s start by telling the truth: machines don’t learn. What a typical “learning machine”\ndoes, is ﬁnding a mathematical formula, which, when applied to a collection of inputs (called\n“training data”), produces the desired outputs. This mathematical formula also generates the\ncorrect outputs for most other inputs (distinct from the training data) on the condition that\nthose inputs come from the same or a similar statistical distribution as the one the training\ndata was drawn from.\nWhy isn’t that learning? Because if you slightly distort the inputs, the output is very likely\nto become completely wrong. It’s not how learning in animals works. If you learned to play\na video game by looking straight at the screen, you would still be a good player if someone\nrotates the screen slightly. A machine learning algorithm, if it was trained by “looking”\nstraight at the screen, unless it was also trained to recognize rotation, will fail to play the\ngame on a rotated screen.'

In [8]:
# instantiate vector store
token = secrets.token_urlsafe(16)
vector_store = VectorStore(documents=splits, token=token)

In [41]:
# langchain uses ChromaDB running in the same process
client = chromadb.PersistentClient(path=vector_store._persist_directory)
collections = client.list_collections()
print(collections)

# langchain creates a single collection named 'langchain' for the pdf file
# each split is stored as a separate document in the collection
collection = client.get_collection(name="langchain")
print(f"Number of documents in collection '{collection.name}': {collection.count()}")

# print metadata of the collection
print(collection.metadata)

['langchain']
Number of documents in collection 'langchain': 385
None


In [42]:
# get first 3 documents from chroma
results = collection.peek(3)
results

{'ids': ['5f1c6037-4a44-448e-8c0a-1eb865f936de',
  '5054509a-38a9-4ee7-a5a7-bec80ea3fd7b',
  '3a3b8e53-926a-4f84-b0af-50b878721e21'],
 'embeddings': array([[-0.02419357,  0.00073245,  0.01287257, ..., -0.03229471,
         -0.03061956, -0.01187709],
        [-0.0436933 ,  0.00408339,  0.00118799, ..., -0.02855287,
         -0.01803412, -0.00582509],
        [-0.02305829, -0.01077303,  0.01361008, ..., -0.01046283,
         -0.02208891, -0.00805877]], shape=(3, 1536)),
 'documents': ['The\nHundred-\nPage\nMachine\nLearning\nBook\nAndriy Burkov',
  '“All models are wrong, but some are useful.”\n— George Box\nThe book is distributed on the “read ﬁrst, buy later” principle.\nAndriy Burkov The Hundred-Page Machine Learning Book - Draft',
  'Preface\nLet’s start by telling the truth: machines don’t learn. What a typical “learning machine”\ndoes, is ﬁnding a mathematical formula, which, when applied to a collection of inputs (called\n“training data”), produces the desired outputs. This mathem

In [9]:
conversational_chain = vector_store.create_conversational_retrieval_chain(k=2)
response = conversational_chain.invoke("Explain logistic regression")
print(response['answer'])

/Users/patrick/projects/doc-chat/api/src/doc_chat/rag/vector_store.py:53: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(


 Logistic regression is a type of classification model that uses a continuous function called the standard logistic function (also known as the sigmoid function) to predict the probability of a certain outcome. This function maps input values to a range between 0 and 1, making it suitable for binary classification problems. By optimizing the values of the function's parameters, we can interpret the output as the probability of a certain outcome being positive. The choice of threshold for determining positive or negative labels may vary depending on the problem. Logistic regression was developed as a simple linear classification model before the advent of computers, and it remains a popular and useful tool in machine learning.


In [10]:
source_docs = response['source_documents']
print("Number of source documents: " + str(len(source_docs)))

for i, doc in enumerate(source_docs):
    print(f"Source document {i}")
    print(f"Page: {doc.metadata['page']}")
    print(f"Content: {doc.page_content[:10]}")
    print("-----")

source_docs

Number of source documents: 2
Source document 0
Page: 32
Content: 3.
By look
-----
Source document 1
Page: 32
Content: Figure 3: 
-----


[Document(metadata={'creationdate': '2018-12-18T05:07:46+00:00', 'creator': 'PyPDF', 'moddate': '2019-01-22T19:51:34+00:00', 'page': 32, 'page_label': '33', 'producer': '3-Heights(TM) PDF Optimization Shell 4.8.25.2 (http://www.pdf-tools.com)', 'source': '../data/pdf/The Hundred-Page Machine Learning Book.pdf', 'total_pages': 152}, page_content='3.\nBy looking at the graph of the standard logistic function, we can see how well it ﬁts our\nclassiﬁcation purpose: if we optimize the values ofx and b appropriately, we could interpret\nthe output off(x) as the probability ofyi being positive. For example, if it’s higher than or\nequal to the threshold0.5 we would say that the class ofx is positive; otherwise, it’s negative.\nIn practice, the choice of the threshold could be di\x00erent depending on the problem. We\nreturn to this discussion in Chapter 5 when we talk about model performance assessment.\nSo our logistic regression model looks like this:\nAndriy Burkov The Hundred-Page Machine

In [11]:
source_docs[0]

Document(metadata={'creationdate': '2018-12-18T05:07:46+00:00', 'creator': 'PyPDF', 'moddate': '2019-01-22T19:51:34+00:00', 'page': 32, 'page_label': '33', 'producer': '3-Heights(TM) PDF Optimization Shell 4.8.25.2 (http://www.pdf-tools.com)', 'source': '../data/pdf/The Hundred-Page Machine Learning Book.pdf', 'total_pages': 152}, page_content='3.\nBy looking at the graph of the standard logistic function, we can see how well it ﬁts our\nclassiﬁcation purpose: if we optimize the values ofx and b appropriately, we could interpret\nthe output off(x) as the probability ofyi being positive. For example, if it’s higher than or\nequal to the threshold0.5 we would say that the class ofx is positive; otherwise, it’s negative.\nIn practice, the choice of the threshold could be di\x00erent depending on the problem. We\nreturn to this discussion in Chapter 5 when we talk about model performance assessment.\nSo our logistic regression model looks like this:\nAndriy Burkov The Hundred-Page Machine 

In [12]:
source_docs[1]

Document(metadata={'creationdate': '2018-12-18T05:07:46+00:00', 'creator': 'PyPDF', 'moddate': '2019-01-22T19:51:34+00:00', 'page': 32, 'page_label': '33', 'producer': '3-Heights(TM) PDF Optimization Shell 4.8.25.2 (http://www.pdf-tools.com)', 'source': '../data/pdf/The Hundred-Page Machine Learning Book.pdf', 'total_pages': 152}, page_content='Figure 3: Standard logistic function.\nAt the time where the absence of computers required scientists to perform manual calculations,\nthey were eager to ﬁnd a linear classiﬁcation model. They ﬁgured out that if we deﬁne a\nnegative label as0 and the positive label as1, we would just need to ﬁnd a simple continuous\nfunction whose codomain is(0, 1). In such a case, if the value returned by the model for\ninput x is closer to0, then we assign a negative label tox; otherwise, the example is labeled\nas positive. One function that has such a property is thestandard logistic function(also\nknown as thesigmoid function):\nf(x)= 1\n1+ e≠x ,\nwhere e i

In [13]:
# conversational_chain can be used to ask follow-up questions
response = conversational_chain.invoke("Give me the formula")
print(response['answer'])

 The formula for the standard logistic function used in logistic regression is f(x)= 1/(1+e^-x).


In [14]:
# qa chain does not use the history of the conversation
qa_chain = vector_store.create_qa_chain(k=2)
response = qa_chain.invoke("Explain logistic regression")
print(response['result'])

 Logistic regression is a type of classification model that uses a continuous function called the standard logistic function (also known as the sigmoid function) to predict the probability of a certain outcome. This function maps input values to a range between 0 and 1, making it suitable for binary classification problems. By optimizing the values of the function's parameters, we can interpret the output as the probability of a certain outcome being positive. The choice of threshold for determining positive or negative labels may vary depending on the problem. Logistic regression was developed as a simple linear classification model before the advent of computers, and it remains a popular and useful tool in machine learning.


In [15]:
# a follow-up question to the qa does not result in answer
response = qa_chain.invoke("Give me the formula")
print(response['result'])

 I don't know the formula.


In [16]:
# retrieve only documents without call to the LLM
documents = vector_store.retrieve_documents("Explain logistic regression", k=2)

for doc in documents:
    print(f"Page: {doc.metadata['page']}")
    print(f"Content: {doc.page_content[:300]}")

/Users/patrick/projects/doc-chat/api/src/doc_chat/rag/vector_store.py:74: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  return retriever.get_relevant_documents(query)


Page: 32
Content: 3.
By looking at the graph of the standard logistic function, we can see how well it ﬁts our
classiﬁcation purpose: if we optimize the values ofx and b appropriately, we could interpret
the output off(x) as the probability ofyi being positive. For example, if it’s higher than or
equal to the thresho
Page: 32
Content: Figure 3: Standard logistic function.
At the time where the absence of computers required scientists to perform manual calculations,
they were eager to ﬁnd a linear classiﬁcation model. They ﬁgured out that if we deﬁne a
negative label as0 and the positive label as1, we would just need to ﬁnd a simp
